In [ ]:
import numpy as np
from PIL import Image

import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision.transforms as transforms
from torchvision import datasets

import os
import torchvision.transforms.functional as TF

from torch.utils.data import random_split, DataLoader

In [ ]:
class RoadDataset(datasets.ImageFolder):
    def __getitem__(self, index):
        # 1. Get the path, image, and label
        path, label_idx = self.samples[index]
        image = self.loader(path) # Loads as PIL Image
        
        filename = os.path.basename(path)
        width, height = image.size

        if "arenacam" in filename:
            # Car hood in image
            image = TF.crop(image, top=700, left=1200, height=400, width=400)
        elif "pylon_camera" in filename:
            # No car hood in view
            image = TF.crop(image, top=400, left=800, height=400, width=400)
        elif width < 400:
            pass #dont crop images that are too small
        else:
            #base case just in case
            image = TF.center_crop(image, (400, 400))

        # apply transforms (resize, tensor, normalize)
        if self.transform is not None:
            image = self.transform(image)

        # Returns: (image, index of label)
        return image, label_idx
    
    def get_classes(self):
        return self.classes




In [ ]:
post_crop_transform = transforms.Compose([
    transforms.Resize(224), #standard image size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

local_path = "C:\ml_images\catergorized2"

full_dataset = RoadDataset(root=local_path, transform=post_crop_transform)

# Test it out
img, label, surface = full_dataset[0]
print(f"Loaded image from folder: {surface} (Label: {label})")

In [ ]:
#Split into train and test
train_size = int(0.9 * len(full_dataset))
test_size = len(full_dataset) - train_size

# Split randomly (using a seed for reproducibility)
train_subset, test_subset = random_split(
    full_dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

#dataloaders
train_loader = DataLoader(
    train_subset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

test_loader = DataLoader(
    test_subset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

NameError: name 'full_dataset' is not defined

In [4]:
#get class name array to assign based on indexes
class_names = full_dataset.get_classes()
print(class_names)

NameError: name 'full_dataset' is not defined

In [ ]:
class NeuralNet(nn.Module):

    #Neural net is default torch neural network, we use super to call parent initialization
    def __init__(self):
        super().__init__()

        #LAYER DEFINITIONS
        
        #Convolutional layer: use a kernel to extract features and information from data
        #Pool layer: simplify image by combining a grid of pixels into one
         
        #originally 3 input channels (RGB), 12 feature maps, 5x5 grid
        self.conv1 = nn.Conv2d(3, 12, 5) #(32 - 5)/stride(1) + 1 = new shape(12 channels, 28x28)
        self.pool = nn.MaxPool2d(2, 2) # (12, 14, 14) pools 2 a 2x2 of pixels into 1 pixel, so shape is divided by 2
        self.conv2 = nn.Conv2d(12, 24, 5) #14-5 / 1 + 1  = (24, 10x10)
        
        #We will flatten in between here
        self.fc1 = nn.Linear(24 * 5 * 5, 120) #After flattening we have 600 neurons we will downsize to 300
        self.fc2 = nn.Linear(120, 66) #Can do whatever you want in here, this is the network vuilding section, we can add relu and other activation functions later
        self.fc3 = nn.Linear(66, 4) #in the end you need 10 layers for the 10 decision options

    #now that we define our cnn functions we just connect the dots with an execute function
    def forward(self, x):
        #CONVOLUTIONAL
        x = self.pool(F.relu(self.conv1(x))) #F.relu is activation function to break linearity
        x = self.pool(F.relu(self.conv2(x)))
        #FLATTEN 
        x = torch.flatten(x, 1) 
        #NEURAL NETWORK
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [55]:
#training definitions
net = NeuralNet()

#use a torch loss function
loss_function = nn.CrossEntropyLoss()
#use a torch stochastic grading descent for optimization
optimizer = optim.SGD(net.parameters(), lr = 0.001, momentum = 0.9) #essentially calculates slope of loss function based on each sample and adjusts parameters/weights

In [ ]:
#train
for epoch in range(30):
    print(f'Training epoch {epoch}...')

    running_loss = 0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad() #set all gradients to zero

        outputs = net(inputs) #have the network predict outputs

        loss = loss_function(outputs, labels) #calculate loss (how far off were our predictions from actuals?)
        loss.backward() #backpropagation calculus (compute gradient) (this is the magic)
        optimizer.step() #take a step (depenedent on learning rate)

        running_loss += loss.item() #assign loss to running loss

    print(f'Loss: {running_loss / len(train_loader):.4f}')
        

Training epoch 0...
Loss: 2.2441
Training epoch 1...
Loss: 1.8063
Training epoch 2...
Loss: 1.5480
Training epoch 3...
Loss: 1.4242
Training epoch 4...
Loss: 1.3247
Training epoch 5...
Loss: 1.2394
Training epoch 6...
Loss: 1.1682
Training epoch 7...
Loss: 1.1098
Training epoch 8...
Loss: 1.0602
Training epoch 9...
Loss: 1.0178
Training epoch 10...
Loss: 0.9821
Training epoch 11...
Loss: 0.9440
Training epoch 12...
Loss: 0.9147
Training epoch 13...
Loss: 0.8800
Training epoch 14...
Loss: 0.8489
Training epoch 15...
Loss: 0.8174
Training epoch 16...
Loss: 0.7911
Training epoch 17...
Loss: 0.7624
Training epoch 18...
Loss: 0.7313
Training epoch 19...
Loss: 0.7101
Training epoch 20...
Loss: 0.6803
Training epoch 21...
Loss: 0.6571
Training epoch 22...
Loss: 0.6388
Training epoch 23...
Loss: 0.6108
Training epoch 24...
Loss: 0.5939
Training epoch 25...
Loss: 0.5692
Training epoch 26...
Loss: 0.5472
Training epoch 27...
Loss: 0.5286
Training epoch 28...
Loss: 0.5067
Training epoch 29...
Los

In [57]:
#export weights/parameters
torch.save(net.state_dict(), 'trained_net.pth')

In [58]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [60]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy}%')

Accuracy: 67.97%
